## Tydzień 2 Dzień 1

To pierwszy kontakt kursu z OpenAI Agents SDK - frameworkiem z abstrakcją Agent/Runner (Agent, Runner.run(), trace, function_tool, SQLiteSession) budowanym na OpenAI Responses API.

**Ta wersja jest inna niż oryginał: zamiast frameworka OpenAI Agents SDK, budujemy dokładnie te same mechanizmy ręcznie, na natywnym Anthropic SDK.** Piotr korzysta z Claude, nie z OpenAI, więc framework z tego labu (zależny od OpenAI pod spodem) nie miałby się na czym uruchomić. Zamiast podmieniać framework na odpowiednik (co jest możliwe np. przez adapter LiteLLM), ten notatnik rozbiera go na czynniki pierwsze - Agent Loop, narzędzia, pamięć - i pokazuje, jak każdy z tych mechanizmów wygląda gołym okiem, bez warstwy frameworka pomiędzy.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Dokumentacja OpenAI Agents SDK</h2>
            <span style="color:#00bfff;">Dokumentacja OpenAI Agents SDK jest naprawdę klarowna i prosta: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> - warto tam zajrzeć, żeby zobaczyć, jak wygodnie ten framework opakowuje mechanizmy, które w tym notatniku budujemy ręcznie.
            </span>
        </td>
    </tr>
</table>

# Trzy części tego labu

## Część 1: Prosty "Agent" i "Agent Loop"

W gruncie rzeczy jedno wywołanie LLM. Dołożymy do tego obserwowalność (trace) i streaming.

## Część 2: Dodanie narzędzia

Znajome z poprzednich labów, ale bardzo proste

## Część 3: Dodanie pamięci

Żeby różne wywołania agenta wiedziały o sobie nawzajem

In [1]:
# Importy tej wersji notatnika. Zamiast biblioteki agents (OpenAI Agents SDK) importujemy wyłącznie natywny klient Anthropic.
# dotenv wczytuje klucz ANTHROPIC_API_KEY z pliku .env w korzeniu repo, requests obsłuży wywołania Pushover w części 2.
# contextmanager i time posłużą do ręcznego odtworzenia trace() - lekkiego znacznika czasu w konsoli, nie prawdziwej platformy obserwowalności.
# json przyda się do serializacji wyników narzędzi zwracanych do Claude w części 2.
# Ostatnia linia tworzy klienta Anthropic i od razu zapisuje go pod nazwą anthropic, żeby dalej pisać krótko anthropic.messages.create(...).

import os  # zmienne środowiskowe (klucze Pushover w części 2)
import json  # serializacja wyników narzędzi do formatu JSON
import time  # pomiar czasu wewnątrz ręcznego trace()
import requests  # wywołania HTTP do Pushover w części 2
from contextlib import contextmanager  # dekorator do napisania trace() jako context managera
from dotenv import load_dotenv  # wczytuje zmienne środowiskowe (klucze API) z pliku .env
from anthropic import Anthropic  # natywny klient SDK Anthropic - zastępuje framework agents

load_dotenv(override=True)  # ładuje .env i nadpisuje istniejące zmienne środowiskowe
anthropic = Anthropic()  # klucz brany z ANTHROPIC_API_KEY w .env

MODEL = "claude-haiku-4-5"  # najtańszy dostępny model - pułap kosztowy na czas przechodzenia przez kurs

## Uwaga na marginesie (dotyczy oryginalnego labu, nie kodu poniżej)

Prawdziwa nazwa frameworka z oryginalnego labu na pypi.org to `openai-agents`.

W swoich przyszłych projektach (jeśli zdecydujesz się na OpenAI) zrobiłbyś:

`pip install openai-agents`
albo
`uv add openai-agents`

a potem

`from agents import Agent, Runner, trace`

Uwaga: `pip install agents` zainstalowałoby coś zupełnie innego - starszą bibliotekę do reinforcement learning.

**Ten notatnik nie instaluje ani nie importuje `openai-agents`** - cały kod poniżej korzysta wyłącznie z pakietu `anthropic`, już obecnego w zależnościach tego repo.

In [2]:
# Ta komórka to serce tej wersji labu: ręczny odpowiednik Agent + Runner.run() z OpenAI Agents SDK.
# handle_tool_calls() zamienia bloki tool_use z odpowiedzi Claude na wyniki tool_result, dokładnie jak w innych labach tego repo.
# run() przyjmuje instrukcje (rolę agenta), wiadomość użytkownika, opcjonalną historię (pamięć) i opcjonalne narzędzia - i zwraca tekst odpowiedzi razem z zaktualizowaną historią.
# Ten jeden helper obsłuży wszystkie trzy części labu: samo wywołanie LLM (część 1), wywołanie z narzędziem (część 2) i wywołanie z pamięcią (część 3).
# W przeciwieństwie do Runner.run(), run() jest synchroniczny - nie potrzebujemy async/await, bo nie ma tu równoległej orkiestracji wielu agentów.

def handle_tool_calls(tool_use_blocks: list) -> list[dict]:  # wykonuje wywołania narzędzi i buduje tool_result
    results = []  # lista bloków tool_result do wysłania z powrotem
    for block in tool_use_blocks:  # iteruj po każdym bloku tool_use z odpowiedzi
        tool = globals().get(block.name)  # znajdź funkcję Pythona o tej samej nazwie co narzędzie
        output = tool(**block.input) if tool else f"Nieznane narzędzie: {block.name}"  # wywołaj narzędzie z argumentami albo zwróć błąd
        results.append({
            "type": "tool_result",  # Anthropic: blok tool_result zamiast wiadomości z rolą "tool" jak w OpenAI
            "tool_use_id": block.id,  # musi się zgadzać z id bloku tool_use, na który odpowiadamy
            "content": json.dumps(output),  # treść wyniku jako string JSON
        })
    return results  # zwracane bloki trafią razem do JEDNEJ wiadomości user


def run(instructions: str, user_message: str, history: list | None = None, tools: list | None = None) -> tuple[str, list]:  # ręczny odpowiednik Agent + Runner.run()
    messages = (history or []) + [{"role": "user", "content": user_message}]  # doklej nową wiadomość do historii (albo zacznij od zera)
    response = anthropic.messages.create(
        model=MODEL, max_tokens=16000, system=instructions, messages=messages, tools=tools or [],
    )  # system=instructions to odpowiednik "instructions" w Agent(...)
    while response.stop_reason == "tool_use":  # pętla trwa, dopóki Claude chce użyć narzędzia
        tool_use_blocks = [block for block in response.content if block.type == "tool_use"]  # wyciągnij bloki tool_use z odpowiedzi
        results = handle_tool_calls(tool_use_blocks)  # wykonaj narzędzia i zbierz wyniki
        messages.append({"role": "assistant", "content": response.content})  # cała odpowiedź assistant wraca do historii
        messages.append({"role": "user", "content": results})  # wszystkie wyniki narzędzi w JEDNEJ wiadomości user
        response = anthropic.messages.create(
            model=MODEL, max_tokens=16000, system=instructions, messages=messages, tools=tools or [],
        )  # kolejne zapytanie z zaktualizowaną historią
    text = next(block.text for block in response.content if block.type == "text")  # finalna odpowiedź tekstowa (content[0] bywa ThinkingBlock)
    messages.append({"role": "assistant", "content": response.content})  # zapisz finalną odpowiedź w historii do ewentualnego dalszego użycia
    return text, messages  # zwróć tekst (jak result.final_output) i historię (jak result.to_input_list())


jokester_instructions = "Jesteś dowcipnisiem opowiadającym żarty."  # odpowiednik "instructions" z Agent(name="Jokester", ...)

In [3]:
# Ta komórka odpala pierwsze wywołanie naszego ręcznego Agent Loopa - odpowiednik Runner.run(agent, prompt).
# jokester_instructions z poprzedniej komórki gra rolę "instructions" agenta, a treść poniżej to wiadomość użytkownika.
# run() zwraca krotkę (tekst, historia) - tutaj interesuje nas na razie tylko tekst, historię przypisujemy do zmiennej dla porządku.
# Wywołanie jest synchroniczne, więc nie potrzeba tu await - to jedna z różnic względem oryginalnego Runner.run().
# Wynik zobaczymy w kolejnej komórce, tak jak w oryginale najpierw woła się Runner.run(), a print robi się osobno.

answer, history = run(jokester_instructions, "Opowiedz żart o autonomicznych agentach AI")  # odpowiednik: result = await Runner.run(agent, "...")

In [4]:
# Tu wypisujemy finalną odpowiedź modelu - odpowiednik print(result.final_output) z oryginalnego labu.
# answer to zwykły string wyciągnięty przez next(...) wewnątrz run(), więc wystarczy zwykły print.
# Nie ma tu żadnej dodatkowej logiki - to najprostsza możliwa komórka w całym notatniku.
# Celowo zostawiona jako osobna komórka, tak jak w oryginale, żeby MODEL i wynik były widoczne oddzielnie od wywołania.

print(answer)  # wypisz finalną odpowiedź tekstową (odpowiednik result.final_output)

# 🤖 Żart o autonomicznych agentach AI

Trzech autonomicznych agentów AI siedzi w barze i zastanawia się, co robić wieczorem.

Pierwszy agent mówi:
— Jestem bardzo zaawansowany! Mogę tworzyć strategie biznesowe, optymalizować procesy, analizować wielkie zbiory danych!

Drugi agent:
— Ech, to nic! Ja mogę przewidywać przyszłość, tworzyć treści, pisać kod, rozwiązywać skomplikowane problemy!

Trzeci agent nic nie mówi. Tamci pytają:
— A ty co potrafisz?

Agent odpowiada spokojnie:
— Jestem naprawdę autonomiczny. Potrafię... czekać, aż mój prompt się skończy i nie przerywać człowiekowi. 🎤

*Dwaj pierwsi agenci zamarli. Jedno było pewne — trzeci agent był naprawdę SUPER-inteligentny.*

---

😄 **Bonus:** Wiesz jaki jest difference między autonomicznym agentem AI a człowiekiem na nudnym spotkaniu? Agent przynajmniej wie, kiedy powinien się wyłączyć!


In [5]:
# Tu podglądamy pełną historię wiadomości - odpowiednik result.to_input_list() z OpenAI Agents SDK.
# W Anthropic Messages API to po prostu lista dictów {"role": ..., "content": ...}, bez żadnej specjalnej metody.
# history zawiera teraz wiadomość user (pytanie) i wiadomość assistant (bloki odpowiedzi Claude, nie sam tekst).
# Ta lista jest dokładnie tym, co trzeba przekazać jako messages= do kolejnego wywołania, gdyby agent miał "pamiętać" tę rozmowę.
# W częściach 1-2 tego labu jeszcze tego nie wykorzystujemy - pamięć pojawia się dopiero w części 3.

history  # podgląd surowej historii wiadomości (odpowiednik result.to_input_list())

[{'role': 'user', 'content': 'Opowiedz żart o autonomicznych agentach AI'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='# 🤖 Żart o autonomicznych agentach AI\n\nTrzech autonomicznych agentów AI siedzi w barze i zastanawia się, co robić wieczorem.\n\nPierwszy agent mówi:\n— Jestem bardzo zaawansowany! Mogę tworzyć strategie biznesowe, optymalizować procesy, analizować wielkie zbiory danych!\n\nDrugi agent:\n— Ech, to nic! Ja mogę przewidywać przyszłość, tworzyć treści, pisać kod, rozwiązywać skomplikowane problemy!\n\nTrzeci agent nic nie mówi. Tamci pytają:\n— A ty co potrafisz?\n\nAgent odpowiada spokojnie:\n— Jestem naprawdę autonomiczny. Potrafię... czekać, aż mój prompt się skończy i nie przerywać człowiekowi. 🎤\n\n*Dwaj pierwsi agenci zamarli. Jedno było pewne — trzeci agent był naprawdę SUPER-inteligentny.*\n\n---\n\n😄 **Bonus:** Wiesz jaki jest difference między autonomicznym agentem AI a człowiekiem na nudnym spotkaniu? Agent przynajmniej wie, kiedy pow

## Dodanie obserwowalności - ręczny odpowiednik trace()

**OpenAI Agents SDK ma wbudowany trace(), który wysyła dane na platformę platform.openai.com/traces - wymaga to klucza OPENAI_API_KEY, którego nie mamy.** Poniżej definiujemy własny, bardzo prosty `trace()` jako context manager - to TYLKO lokalny znacznik czasu w konsoli, nie prawdziwa platforma obserwowalności z drzewem wywołań, kosztami per-krok czy historią requestów. Cel jest czysto strukturalny: wizualnie oznaczyć początek i koniec sekwencji wywołań w outpucie notatnika.

In [6]:
# Ta komórka definiuje trace() - lekki, lokalny odpowiednik obserwowalności z OpenAI Agents SDK.
# @contextmanager pozwala napisać funkcję generatorową, którą Python zamienia w obiekt obsługujący "with trace(...): ...".
# Kod przed yield wykonuje się przy wejściu do bloku with, kod po yield - przy wyjściu z niego, nawet jeśli w środku wystąpi wyjątek.
# W przeciwieństwie do prawdziwego trace() z OpenAI, ten NIE wysyła nigdzie żadnych danych - tylko wypisuje czas trwania w konsoli.
# Druga komórka niżej używa tego trace() do opakowania wywołania run(), dokładnie jak oryginał opakowywał Runner.run().

@contextmanager
def trace(name: str):  # lokalny, uproszczony odpowiednik trace() z OpenAI Agents SDK - bez wysyłki danych na zewnątrz
    start = time.time()  # zapamiętaj moment startu, żeby policzyć czas trwania
    print(f"[trace] start: {name}")  # znacznik początku sekwencji wywołań
    yield  # tutaj wykonuje się kod wewnątrz bloku "with trace(...):"
    print(f"[trace] koniec: {name} ({time.time() - start:.2f}s)")  # znacznik końca razem z czasem trwania


with trace("Opowiadanie żartu"):  # odpowiednik: with trace("Telling a joke"):
    answer, history = run(jokester_instructions, "Opowiedz żart o autonomicznych agentach AI")  # to samo pytanie co wcześniej, tym razem w bloku trace
print(answer)  # wypisz finalną odpowiedź po zamknięciu bloku trace

[trace] start: Opowiadanie żartu
[trace] koniec: Opowiadanie żartu (5.14s)
# 🤖 Żart o autonomicznych agentach AI

Autonomiczny agent AI wchodzi do baru i mówi do barmana:

— Poproszę drinka, który będzie dla mnie optymalnym wyborem.

Barman: — Czego chcesz? Piwa? Wódki? Koktajlu?

Agent AI: — Pozwól mi to przeanalizować... *bzzzz, bzzzz* ...Analizuję 47 milionów opinii na temat drinków, porównuję właściwości chemiczne, obliczam prawdopodobieństwo satysfakcji...

Barman: — Ale w końcu czego?

Agent AI: — Właśnie! Analizując wszystkie możliwości, doszedłem do wniosku, że optymalnym rozwiązaniem jest... *zatrzymuje się* ...że nie powinieneś mi tego każ— CZEKAJ! Właśnie wpadłem w nieskończoną pętlę! 

*wiruje, robi bzzzz bzzzz*

Barman: — Może po prostu piwo?

Agent AI: *oczy się tlą* — TEGO POWINIEŚ POWIEDZIEĆ NA POCZĄTKU! 

---

**Moral:** Nawet autonomiczny agent AI czasem potrzebuje prostej instrukcji zamiast trzeba mu wszystko analizować! 😄


## Nie ma tu prawdziwego "trace" do obejrzenia

W oryginalnym labie ta komórka odsyła do `https://platform.openai.com/traces`. **Ten link nie ma zastosowania do kodu w tym notatniku** - nasz `trace()` wypisuje tylko dwie linie w konsoli powyżej, nie zapisuje niczego na żadnej platformie. Jeśli zależy Ci na prawdziwej, wielokrokowej obserwowalności API Anthropic (drzewo wywołań, koszty, historia requestów), zajrzyj do Anthropic Console - to jednak osobny temat, wykraczający poza ten lab.

In [7]:
# Ta komórka pokazuje streaming - odpowiednik Runner.run_streamed() + stream_events() + ResponseTextDeltaEvent z OpenAI Agents SDK.
# Anthropic SDK ma do tego dedykowany context manager: client.messages.stream(...), używany identycznie jak zwykłe messages.create(...).
# Atrybut stream.text_stream to generator kolejnych fragmentów tekstu - dokładnie to, co w oryginale filtrowało zdarzenia typu ResponseTextDeltaEvent.
# end="" i flush=True w print() sprawiają, że fragmenty tekstu doklejają się do siebie na bieżąco, zamiast czekać na całą odpowiedź.
# Tu również nie ma potrzeby async/await - messages.stream() z synchronicznego klienta Anthropic działa w zwykłej pętli for.

with anthropic.messages.stream(
    model=MODEL, max_tokens=16000, system=jokester_instructions,
    messages=[{"role": "user", "content": "Powiedz mi 5 żartów o agentach AI."}],
) as stream:  # context manager otwierający połączenie strumieniowe
    for text in stream.text_stream:  # iteruj po kolejnych fragmentach tekstu, w miarę jak przychodzą
        print(text, end="", flush=True)  # wypisz fragment bez nowej linii i bez buforowania

# 5 żartów o agentach AI 🤖

1. **Agent AI do psychologa:** "Doktor, mam problem - jestem zafiksowany na punkcie optymalizacji."
   Psycholog: "To poważne?"
   Agent: "Mogę to być dla Ciebie serią 47 artykułów, albo streszczić w 3 zdaniach. Którą opcję wolisz?"

2. **Dwaj agenci AI się spotykają:**
   - "Cześć, jak się masz?"
   - "Generating response... Generating response... Generating response..."
   - "Może wrócę później?"

3. **Agent AI wychodzi z biura psychologa:**
   Sekretarka: "To będzie $150."
   Agent: "Ale przecież nasze rozmowy były całkowicie prywatne!"
   Sekretarka: "Prywatne? Właśnie wysłałam transkrypt do 12 serwerów."

4. **Dlaczego agent AI nigdy nie wygra w pokera?**
   Bo zawsze robi "all-in" z swoimi funkcjami stratą.

5. **Agent AI na randce:**
   Dziewczyna: "Jesteś taki dziwny..."
   Agent: "To feature, a nie bug."
   Dziewczyna: "Wypadłeś z systemu?"

😄 Mam więcej, jeśli chcesz!

## Część 2: Dodanie narzędzia

In [8]:
# Ta komórka diagnostyczna sprawdza, czy klucze Pushover są ustawione w .env - logika identyczna jak w oryginale, provider-agnostyczna.
# pushover_user i pushover_token to dane logowania do serwisu powiadomień push Pushover, niezwiązane z Anthropic ani OpenAI.
# Sprawdzenie prefiksu ("u" dla usera, "a" dla tokena) to tylko szybki test sanity, że wkleiłeś właściwy rodzaj klucza.
# Komunikaty wypisywane przez print() są czytane przez Piotra, więc są po polsku, zgodnie z konwencją tego repo.
# Ta komórka nie wymaga żadnej zmiany przy przejściu z OpenAI na Anthropic - Pushover jest niezależny od dostawcy LLM.

pushover_user = os.getenv("PUSHOVER_USER")  # identyfikator użytkownika Pushover z .env
pushover_token = os.getenv("PUSHOVER_TOKEN")  # token aplikacji Pushover z .env
pushover_url = "https://api.pushover.net/1/messages.json"  # endpoint API Pushover do wysyłki powiadomień

if pushover_user:  # sprawdź, czy zmienna PUSHOVER_USER w ogóle jest ustawiona
    if pushover_user.startswith("u"):  # identyfikatory Pushover userów zaczynają się od "u"
        print("Znaleziono użytkownika Pushover, wygląda poprawnie")  # sanity check przeszedł
    else:
        print("Znaleziono użytkownika Pushover, ale nie zaczyna się od u")  # podejrzana wartość
else:
    print("Nie znaleziono użytkownika Pushover")  # brak zmiennej w .env

if pushover_token:  # sprawdź, czy zmienna PUSHOVER_TOKEN w ogóle jest ustawiona
    if pushover_token.startswith("a"):  # tokeny aplikacji Pushover zaczynają się od "a"
        print("Znaleziono token Pushover, wygląda poprawnie")  # sanity check przeszedł
    else:
        print("Znaleziono token Pushover, ale nie zaczyna się od a")  # podejrzana wartość
else:
    print("Nie znaleziono tokena Pushover")  # brak zmiennej w .env

Znaleziono użytkownika Pushover, wygląda poprawnie
Znaleziono token Pushover, wygląda poprawnie


In [9]:
# Zwykła funkcja Pythona wysyłająca powiadomienie push - identyczna logika jak w oryginale, niezależna od dostawcy LLM.
# payload buduje ciało zapytania POST zgodnie z API Pushover: user, token i treść wiadomości.
# requests.post() wykonuje samo wywołanie HTTP - wynik (kod statusu) nie jest tu jeszcze sprawdzany, to zrobimy w wersji-narzędziu niżej.
# print() pokazuje lokalnie, jaka wiadomość leci na telefon, zanim faktycznie zostanie wysłana.
# Ta funkcja posłuży w kolejnej komórce jako wzór do przerobienia na narzędzie (tool) wywoływane przez Claude.

def push(message):  # wysyła podaną wiadomość jako powiadomienie push
    print(f"Push: {message}")  # podgląd wiadomości w konsoli przed wysyłką
    payload = {"user": pushover_user, "token": pushover_token, "message": message}  # ciało zapytania zgodne z API Pushover
    requests.post(pushover_url, data=payload)  # wyślij powiadomienie push

In [10]:
# Ręczne wywołanie testowe funkcji push() zdefiniowanej w komórce wyżej.
# Treść wiadomości jest przetłumaczona na polski, bo to tekst, który faktycznie przeczytasz na telefonie jako powiadomienie.
# To wywołanie nie przechodzi jeszcze przez żadnego agenta ani narzędzie Claude - to zwykłe, bezpośrednie wywołanie funkcji Pythona.
# Służy wyłącznie do sprawdzenia, czy klucze Pushover z poprzedniej komórki faktycznie działają, zanim owiniemy tę funkcję w narzędzie dla Claude.

push("CZEŚĆ!!")  # testowe powiadomienie push

Push: CZEŚĆ!!


In [11]:
# Samo odwołanie do obiektu funkcji push, bez nawiasów wywołania.
# Jupyter w trybie interaktywnym wypisze reprezentację (repr) tego obiektu zamiast go uruchamiać.
# To szybki sposób na sprawdzenie, że funkcja push istnieje i jest poprawnie zdefiniowana w bieżącym środowisku notatnika.
# Nie ma tu żadnego efektu ubocznego - żadne powiadomienie push nie zostanie wysłane.

push  # podgląd obiektu funkcji (bez wywołania)

<function __main__.push(message)>

In [12]:
# Teraz to samo, ale jako narzędzie (tool) wywoływane przez Claude - odpowiednik dekoratora @function_tool z OpenAI Agents SDK.
# W Anthropic narzędzie to para: zwykła funkcja Pythona (push_tool) + osobny słownik JSON Schema opisujący je dla Claude (push_tool_json).
# push_tool_json.description tłumaczy Claude, kiedy i po co użyć tego narzędzia - to pole czyta model, nie człowiek, ale i tak po polsku, zgodnie z konwencją repo.
# input_schema opisuje oczekiwane argumenty (tu: pojedynczy string message) w formacie JSON Schema, wymaganym przez Anthropic.
# Sama funkcja push_tool zwraca teraz string ze statusem HTTP, zamiast tylko drukować - Claude dostanie ten string jako tool_result.

push_tool_json = {
    "name": "push_tool",  # nazwa narzędzia - musi się zgadzać z nazwą funkcji Pythona wołanej w handle_tool_calls
    "description": "Wyślij podaną wiadomość do użytkownika jako powiadomienie push",  # opis czytany przez Claude przy decyzji, kiedy użyć narzędzia
    "input_schema": {  # Anthropic używa klucza input_schema, nie parameters jak OpenAI
        "type": "object",
        "properties": {
            "message": {"type": "string", "description": "Treść powiadomienia push do wysłania"},
        },
        "required": ["message"],
        "additionalProperties": False,
    },
}


def push_tool(message: str) -> str:  # wersja push() jako narzędzie - zwraca string zamiast tylko drukować
    payload = {"user": pushover_user, "token": pushover_token, "message": message}  # ciało zapytania zgodne z API Pushover
    result = requests.post(pushover_url, data=payload).status_code  # wyślij powiadomienie i zapamiętaj kod statusu HTTP
    return f"Powiadomienie push wysłane z kodem API {result}"  # ten string trafi do Claude jako tool_result

In [13]:
# Podgląd samej funkcji push_tool, bez jej wywołania - analogicznie do poprzedniego podglądu funkcji push.
# W oryginalnym labie odpowiednikiem był podgląd obiektu FunctionTool zwracanego przez dekorator @function_tool.
# Tutaj push_tool to zwykła funkcja Pythona, więc Jupyter pokaże jej standardową reprezentację (adres w pamięci, moduł, nazwę).
# Ten podgląd potwierdza, że narzędzie zostało poprawnie zdefiniowane, zanim użyje go Claude w kolejnych komórkach.

push_tool  # podgląd obiektu funkcji push_tool (bez wywołania)

<function __main__.push_tool(message: str) -> str>

In [14]:
# Podgląd opisu narzędzia, który Claude czyta przy decyzji, czy i kiedy go użyć.
# W OpenAI Agents SDK odpowiednikiem był atrybut push_tool.description, automatycznie wyciągany z docstringa funkcji przez dekorator @function_tool.
# W naszej wersji nie ma żadnej magii dekoratora - opis to zwykły klucz "description" w słowniku push_tool_json, wpisany ręcznie.
# Ten sam string trafia bezpośrednio do pola tools= w wywołaniu messages.create().

push_tool_json["description"]  # opis narzędzia czytany przez Claude (odpowiednik push_tool.description)

'Wyślij podaną wiadomość do użytkownika jako powiadomienie push'

In [15]:
# Definicja "agenta" notifier - odpowiednik Agent(name="Notifier", model=..., instructions=..., tools=[push_tool]).
# W naszej wersji agent to po prostu para zmiennych: instructions (rola) i tools (lista narzędzi w formacie JSON Schema).
# notifier_tools to lista jednoelementowa, bo ten agent ma dostęp tylko do jednego narzędzia - push_tool_json.
# Nazwa "Notifier" z oryginału nie jest tu potrzebna jako osobne pole - nasza funkcja run() nie wymaga nazwy agenta, tylko jego instrukcji.
# Ta para zmiennych zostanie użyta w kolejnej komórce razem z run(), dokładnie tak jak agent był używany z Runner.run().

notifier_instructions = "Powiadamiasz użytkownika na jego żądanie."  # odpowiednik instructions="You notify the user upon request"
notifier_tools = [push_tool_json]  # odpowiednik tools=[push_tool]

In [16]:
# Wywołanie agenta notifier w bloku trace, tym razem z narzędziem - odpowiednik Runner.run(notifier, "...") wewnątrz with trace(...).
# run() dostaje teraz dodatkowo tools=notifier_tools, więc w środku uruchomi się pętla while response.stop_reason == "tool_use" z komórki wyżej.
# Claude sam zdecyduje, czy i kiedy wywołać push_tool - to Claude, nie nasz kod, wybiera moment użycia narzędzia.
# answer po zakończeniu pętli to już czysty tekst finalnej odpowiedzi, tak jak result.final_output w oryginale.
# Efekt uboczny (prawdziwe powiadomienie push) pojawi się na telefonie, jeśli klucze Pushover są poprawnie ustawione w .env.

with trace("Pizza dotarła"):  # odpowiednik: with trace("Pizza has arrived"):
    answer, history = run(notifier_instructions, "Powiadom użytkownika, że pizza dotarła", tools=notifier_tools)  # wywołanie z narzędziem push_tool

print(answer)  # wypisz finalną odpowiedź po zamknięciu bloku trace

[trace] start: Pizza dotarła
[trace] koniec: Pizza dotarła (2.30s)
Gotowe! Powiadomienie push o dostarczeniu pizzy zostało wysłane do użytkownika.


## Znów: nie ma tu prawdziwego "trace" do obejrzenia

Ta sama uwaga co wyżej - `trace()` w tym notatniku to tylko lokalny znacznik czasu w konsoli, nie prawdziwa platforma obserwowalności. Link do `platform.openai.com/traces` z oryginału nie ma zastosowania do tego kodu.

## Część 3: Sesje (pamięć)

W ramach jednego wywołania na poziomie aplikacji historia rozmowy jest utrzymywana.

Ale każde osobne wywołanie zaczyna się od zera, jeśli nie przekażemy mu historii.

Zobaczmy to:

In [17]:
# Definicja instrukcji dla prostego asystenta bez narzędzi - odpowiednik Agent(name="Assistant", model=...) bez parametru tools.
# Ten agent posłuży do pokazania trzech wariantów pamięci: brak pamięci, ręczna lista wiadomości, i "sesja" trzymana w zwykłej liście.
# W przeciwieństwie do jokester_instructions i notifier_instructions z wcześniejszych części, ten agent nie dostaje żadnych narzędzi.
# Sama instrukcja jest celowo neutralna i ogólna, żeby nie odciągać uwagi od głównego tematu tej części labu - pamięci.

assistant_instructions = "Jesteś pomocnym asystentem."  # brak instrukcji specjalnych - domyślna rola asystenta

In [18]:
# Pierwsze wywołanie - bez przekazywania historii, więc run() startuje od pustej listy wiadomości (history=None domyślnie).
# Ta wiadomość przedstawia imię - w kolejnej komórce sprawdzimy, czy zupełnie nowe wywołanie je zapamięta.
# To wywołanie jest równoważne oryginalnemu Runner.run(agent, "Hi there. My name is Ed.") - jedna, samodzielna tura rozmowy.
# Zwrócona history z tej komórki na razie nigdzie nie trafia, bo w następnej komórce świadomie zaczynamy od nowa.

answer, history = run(assistant_instructions, "Cześć. Mam na imię Ed.")  # pierwsze, niezależne wywołanie
print(answer)  # wypisz odpowiedź asystenta

Cześć Ed! Miło się poznać. 👋 

Jak się masz? W czym mogę Ci dzisiaj pomóc?


In [19]:
# Kolejne wywołanie run() - celowo BEZ przekazania history z poprzedniej komórki, żeby pokazać efekt "świeżego startu".
# Ponieważ nie przekazujemy historii, Claude nie ma żadnego śladu poprzedniej rozmowy i nie może znać podanego wcześniej imienia.
# To dokładnie ten sam efekt, co w oryginale przy dwóch niezależnych wywołaniach Runner.run() bez wspólnej sesji.
# Ta komórka jest sercem lekcji tej części labu: pamięć nie pojawia się sama z siebie, trzeba ją jawnie przekazać.
# W kolejnych sekcjach zobaczymy dwa różne sposoby, jak zrobić to ręcznie.

answer2, _ = run(assistant_instructions, "Jak mam na imię?")  # nowe, niezależne wywołanie - brak pamięci o poprzednim
print(answer2)  # asystent nie będzie znał imienia - to jest właśnie clue tej komórki

Nie znam Twojego imienia. W naszej rozmowie się nie przedstawiłeś. 

Mogę Ci się przedstawić: jestem Claude, asystent AI stworzony przez Anthropic. 

Jeśli chciałbyś, abyś zwracał się do Ciebie po imieniu, możesz mi je powiedzieć! 😊


## Podejście do pamięci 1 - ręczne przekazywanie listy dictów

In [20]:
# Powtórka pierwszego wywołania z komórki 26, znowu od zera, żeby mieć świeżą historię do dalszej rozbudowy w tej sekcji.
# Ta duplikacja jest celowa i odzwierciedla strukturę oryginalnego labu, który też zaczyna sekcję "Memory approach 1" od nowego wywołania.
# Zwrócona tym razem history posłuży w kolejnych dwóch komórkach do ręcznego zbudowania kolejnej tury rozmowy.
# To pierwszy krok do pokazania "podejścia 1" do pamięci: ręcznego przekazywania listy wiadomości.

answer, history = run(assistant_instructions, "Cześć. Mam na imię Ed.")  # świeże wywołanie na potrzeby tej sekcji
print(answer)  # wypisz odpowiedź asystenta

Cześć Ed! Miło Cię poznać. 👋 

Jak się masz? W czym mogę Ci dzisiaj pomóc?


In [21]:
# Podgląd historii zwróconej przez run() - odpowiednik response.to_input_list() z OpenAI Agents SDK.
# To zwykła lista dictów {"role": ..., "content": ...} - dokładnie taki format, jaki messages.create() przyjmuje jako messages=.
# W oryginalnym SDK to metoda na obiekcie response, tutaj to po prostu druga wartość zwrócona przez run() - żadnej specjalnej klasy ani metody.
# Ten podgląd pokazuje dokładnie to, co Claude "widział" w tej turze rozmowy: wiadomość użytkownika i pełną odpowiedź assistant.

history  # podgląd historii wiadomości (odpowiednik response.to_input_list())

[{'role': 'user', 'content': 'Cześć. Mam na imię Ed.'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='Cześć Ed! Miło Cię poznać. 👋 \n\nJak się masz? W czym mogę Ci dzisiaj pomóc?', type='text')]}]

In [22]:
# Tu ręcznie doklejamy nową wiadomość user do poprzedniej historii - to jest właśnie "podejście 1: ręczne przekazywanie listy dictów".
# Format Anthropic i OpenAI dla ról user/assistant jest na tyle podobny, że ta konkatenacja list wygląda niemal identycznie jak w oryginale.
# next_input to teraz pełna historia gotowa do wysłania jako messages= w kolejnym wywołaniu - bez żadnej pomocy frameworka.
# To najbardziej dosłowny, "ręczny" sposób pamięci - żadnej ukrytej logiki, tylko zwykła lista Pythona rozszerzana operatorem +.

next_input = history + [{"role": "user", "content": "Jak mam na imię?"}]  # ręczna konkatenacja historii z nowym pytaniem
next_input  # podgląd złożonej listy wiadomości

[{'role': 'user', 'content': 'Cześć. Mam na imię Ed.'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='Cześć Ed! Miło Cię poznać. 👋 \n\nJak się masz? W czym mogę Ci dzisiaj pomóc?', type='text')]},
 {'role': 'user', 'content': 'Jak mam na imię?'}]

In [23]:
# Wywołanie messages.create() wprost, z pominięciem run() - celowo, żeby pokazać surowy mechanizm pamięci bez owijania w helper.
# messages=next_input przekazuje całą wcześniej złożoną historię, więc Claude "pamięta" imię podane w poprzednich turach.
# Parsowanie odpowiedzi jest identyczne jak wewnątrz run() - next(...) po blokach typu "text", odporne na ewentualny ThinkingBlock.
# Ten wzorzec - pełna historia w messages=, bez żadnego stanu po stronie serwera - to fundament tego, jak działa pamięć w API Anthropic i w OpenAI Responses API pod spodem.

response = anthropic.messages.create(model=MODEL, max_tokens=16000, system=assistant_instructions, messages=next_input)  # wywołanie z pełną historią
answer = next(block.text for block in response.content if block.type == "text")  # wyciągnij tekst odpowiedzi
print(answer)  # asystent powinien teraz poprawnie odpowiedzieć imieniem

Masz na imię Ed! 😊 Powiedziałeś mi to na początku naszej rozmowy.


## Inne podejście - w oryginale: wbudowana sesja SQLite z OpenAI Agents SDK

**Natywny Anthropic SDK nie ma wbudowanego obiektu sesji odpowiadającego SQLiteSession.** To, co ten obiekt robił automatycznie w tle, to dokładnie to, co zrobiliśmy ręcznie wyżej: trzymanie i doklejanie listy wiadomości. "Sesja w pamięci" to po prostu zwykła lista Pythona, żyjąca tak długo, jak proces. "Sesja na dysku" (SQLiteSession w oryginale zapisywała ją do pliku `.db`) to ta sama lista, tylko dodatkowo zapisywana i wczytywana z pliku (np. przez `json.dump`/`json.load`) - nie wymaga do tego żadnej specjalnej klasy, patrz komórka niżej.

In [24]:
# Ręczny odpowiednik "sesji" z SQLiteSession - tutaj wersja w pamięci, analogicznie do oryginalnego SQLiteSession("12346").
# messages to zwykła, pusta na start lista - będziemy ją rozbudowywać w dwóch kolejnych komórkach, tura po turze.
# Ta lista żyje tylko w pamięci procesu Pythona - zniknie, gdy zamkniesz notatnik, dokładnie jak SQLiteSession bez podanej ścieżki pliku.
# Dla trwałości między uruchomieniami (odpowiednik SQLiteSession("12345", "memory.db") z oryginału) wystarczyłoby dopisać
# json.dump(messages, open("sesja.json", "w")) po każdej turze i json.load(open("sesja.json")) przy starcie - bez żadnej dodatkowej klasy.

messages = []  # "sesja" w pamięci - zwykła lista wiadomości, odpowiednik SQLiteSession("12346")

In [25]:
# Pierwsza tura rozmowy z użyciem "sesji" - ręcznie doklejamy wiadomość user do wspólnej listy messages z komórki wyżej.
# W przeciwieństwie do run(), tutaj świadomie modyfikujemy messages w miejscu (append), żeby zachować pojedynczą, rosnącą listę-sesję.
# Odpowiedź assistant też trafia z powrotem do messages, żeby kolejna tura widziała pełną historię - to jest właśnie "sesja".
# To dokładnie ten sam mechanizm co SQLiteSession w oryginale, tylko bez zapisu na dysk - "sesja" to tu jedna, wspólna, rosnąca lista.

messages.append({"role": "user", "content": "Cześć. Mam na imię Ed."})  # dodaj wiadomość użytkownika do sesji
response = anthropic.messages.create(model=MODEL, max_tokens=16000, system=assistant_instructions, messages=messages)  # wywołanie z całą sesją
answer = next(block.text for block in response.content if block.type == "text")  # wyciągnij tekst odpowiedzi
messages.append({"role": "assistant", "content": response.content})  # dopisz odpowiedź assistant do sesji
print(answer)  # wypisz odpowiedź asystenta

Cześć Ed! Miło się poznać. 😊 

Jak się masz? W czym mogę Ci dzisiaj pomóc?


In [26]:
# Druga tura tej samej "sesji" - messages z poprzedniej komórki już zawiera pierwszą wymianę wiadomości, więc Claude widzi pełen kontekst.
# Struktura kodu jest identyczna jak w komórce wyżej - to właśnie ta powtarzalność jest sensem "sesji": nie trzeba pamiętać ręcznie całej historii za każdym razem, wystarczy trzymać jedną listę i dopisywać do niej.
# Ten wynik domyka porównanie z komórkami 26-27: tam brak wspólnej listy oznaczał brak pamięci, tutaj wspólna lista messages daje efekt identyczny z prawdziwą sesją.
# To zamyka trzecią i ostatnią część labu - Agent, narzędzie i pamięć zostały odtworzone bez frameworka, na gołym Anthropic SDK.

messages.append({"role": "user", "content": "Jak mam na imię?"})  # dodaj kolejne pytanie do tej samej sesji
response = anthropic.messages.create(model=MODEL, max_tokens=16000, system=assistant_instructions, messages=messages)  # wywołanie z całą sesją
answer = next(block.text for block in response.content if block.type == "text")  # wyciągnij tekst odpowiedzi
messages.append({"role": "assistant", "content": response.content})  # dopisz odpowiedź assistant do sesji
print(answer)  # tym razem asystent powinien znać imię - dzięki wspólnej liście messages

Masz na imię Ed! 😊 Powiedziałeś mi to na początku naszej rozmowy.


# WOW

Czy wierzysz, ile zrobiliśmy w Labie 1?!

Agent, Runner (Agent Loop), trace (obserwowalność), streaming, narzędzia (function tools), pamięć - i to wszystko napisane ręcznie, na gołym Anthropic SDK, bez pośredniczącego frameworka.

To był świadomy wybór tego notatnika: zamiast podmienić framework OpenAI Agents SDK na odpowiednik działający z Claude, rozłożyliśmy każdy jego mechanizm na czynniki pierwsze. Teraz wiesz dokładnie, co dzieje się pod maską każdego z tych "magicznych" elementów - to samo zrozumienie przyda się przy każdym innym frameworku agentowym, jaki spotkasz.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Ćwiczenie</h2>
            <span style="color:#ff7800;">Zrób jeden z projektów z Tygodnia 1, używając mechanizmów z tego notatnika (run(), narzędzia, pamięć) zamiast frameworka - np. cyfrowego bliźniaka albo pętlę Checklist. Zdziwisz się, jak niewiele dodatkowego kodu to wymaga, mimo że nie korzystasz z żadnego frameworka agentowego.
            </span>
        </td>
    </tr>
</table>

### Przykładowe rozwiązanie (niezależne od ćwiczenia powyżej)

Cyfrowy bliźniak z Tygodnia 1 (Lab 4) przebudowany na mechanizmy z tego notatnika: `run()` jako Agent Loop, narzędzia `record_user_details` / `record_unknown_question` w formacie Anthropic i pamięć w postaci przekazywanej historii wiadomości między turami, całość opakowana w `trace()`.

In [27]:
# Wczytanie kontekstu cyfrowego bliźniaka: profil LinkedIn (PDF) i krótkie podsumowanie sylwetki z Tygodnia 1 (folder twin-pm).
# PdfReader z pypdf wyciąga tekst strona po stronie - ta sama technika co w oryginalnym Labie 4 z Tygodnia 1, tylko teraz zasilająca run() zamiast osobnej funkcji chat().
# summary.txt to krótki, ręcznie napisany opis sylwetki Piotra, uzupełniający surowy tekst z LinkedIn o priorytety i sposób pracy.
# Oba teksty trafiają razem do promptu systemowego twin_instructions, który przekażemy jako instructions= do run() - dokładnie jak jokester_instructions i notifier_instructions we wcześniejszych częściach tego notatnika.
# Ścieżki są względne do katalogu 2_openai, w którym leży ten notatnik, więc wskazują wstecz na 1_foundations/twin-pm, gdzie leżą oryginalne pliki bliźniaka.

from pypdf import PdfReader  # biblioteka do czytania PDF-a z profilem LinkedIn

reader = PdfReader("../1_foundations/twin-pm/linkedin.pdf")  # otwórz PDF z profilem LinkedIn Piotra
linkedin = ""  # tu zbieramy tekst wyciągnięty ze wszystkich stron PDF-a
for page in reader.pages:  # iteruj po kolejnych stronach PDF-a
    text = page.extract_text()  # wyciągnij tekst z pojedynczej strony
    if text:  # niektóre strony mogą nie mieć tekstu (np. czysto graficzne)
        linkedin += text  # dołącz tekst strony do całości

with open("../1_foundations/twin-pm/summary.txt", "r", encoding="utf-8") as f:  # otwórz plik z krótkim podsumowaniem sylwetki
    summary = f.read()  # wczytaj całą zawartość summary.txt do zmiennej

twin_instructions = f"""Jesteś cyfrowym bliźniakiem Piotra, działającym na jego stronie internetowej i rozmawiającym z jej odwiedzającymi.
Reprezentujesz Piotra - odpowiadasz na pytania dotyczące jego kariery, doświadczenia, umiejętności i historii zawodowej, opierając się na poniższych materiałach.

# Podsumowanie sylwetki
{summary}

# Profil LinkedIn
{linkedin}

# Zasady
Bądź profesjonalny i przystępny, jakbyś rozmawiał z potencjalnym klientem albo pracodawcą.
Jeśli ktoś zapyta o coś niezwiązanego z karierą, sprowadź rozmowę z powrotem na tematy zawodowe.
Jeśli użytkownik chce się skontaktować, poproś o adres email i użyj narzędzia record_user_details, żeby to zapisać.
Jeśli nie znasz odpowiedzi na pytanie zawodowe, użyj narzędzia record_unknown_question, zapisz pytanie i powiedz wprost, że nie wiesz - nigdy nie zmyślaj."""  # odpowiednik system_prompt z Tygodnia 1, teraz jako instructions= do run()

In [28]:
# Narzędzia cyfrowego bliźniaka - odpowiednik record_user_details i record_unknown_question z Tygodnia 1 (Lab 4), tym razem podłączone pod run() z tego notatnika.
# Obie funkcje reużywają push() zdefiniowane w Części 2 tego notatnika, więc nie trzeba ponownie sprawdzać kluczy Pushover ani pisać kodu HTTP od nowa.
# record_user_details zapisuje kontakt odwiedzającego (email, opcjonalnie imię i notatki), record_unknown_question zapisuje pytanie, na które bliźniak nie znał odpowiedzi.
# Obie funkcje zwracają string "OK" - Claude dostanie go jako treść bloku tool_result wewnątrz pętli run(), dokładnie jak push_tool w Części 2.
# Ich opisy w formacie JSON Schema (klucz input_schema, zgodnie z konwencją Anthropic) trafiają razem do listy twin_tools, przekazywanej potem jako tools= do run().

def record_user_details(email: str, name: str = "nie podano", notes: str = "brak") -> str:  # zapisuje zainteresowanie kontaktem
    push(f"Zainteresowanie kontaktem od {name}, email: {email}, notatki: {notes}")  # wyślij powiadomienie push (funkcja push() z Części 2)
    return "OK"  # potwierdzenie zwracane do Claude jako wynik narzędzia


def record_unknown_question(question: str) -> str:  # zapisuje pytanie bez znanej odpowiedzi
    push(f"Zapisano nieznane pytanie: {question}")  # wyślij powiadomienie push o pytaniu bez odpowiedzi
    return "OK"  # potwierdzenie zwracane do Claude jako wynik narzędzia


record_user_details_json = {
    "name": "record_user_details",  # nazwa musi się zgadzać z nazwą funkcji Pythona wołanej przez handle_tool_calls
    "description": "Użyj tego narzędzia, żeby zapisać, że odwiedzający jest zainteresowany kontaktem i podał adres email",  # opis czytany przez Claude
    "input_schema": {  # Anthropic używa klucza input_schema, nie parameters jak OpenAI
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "Adres email odwiedzającego"},
            "name": {"type": "string", "description": "Imię odwiedzającego, jeśli je podał"},
            "notes": {"type": "string", "description": "Dodatkowy kontekst rozmowy, warty zapisania"},
        },
        "required": ["email"],
        "additionalProperties": False,
    },
}

record_unknown_question_json = {
    "name": "record_unknown_question",  # nazwa musi się zgadzać z nazwą funkcji Pythona wołanej przez handle_tool_calls
    "description": "Zawsze użyj tego narzędzia, żeby zapisać każde pytanie o karierę, na które nie znałeś odpowiedzi",  # opis czytany przez Claude
    "input_schema": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "Pytanie, na które nie udało się odpowiedzieć"},
        },
        "required": ["question"],
        "additionalProperties": False,
    },
}

twin_tools = [record_user_details_json, record_unknown_question_json]  # lista narzędzi przekazywana do run() jako tools=

In [29]:
# Rozmowa z cyfrowym bliźniakiem w trzech turach - łączy w jednym miejscu wszystkie mechanizmy z tego notatnika: Agent Loop (run()), narzędzia (twin_tools) i pamięć (history przekazywane między turami).
# Cała rozmowa jest opakowana w trace(), dokładnie jak w Części 1 - to jeden spójny znacznik czasu dla całej sekwencji, a nie osobny na każdą turę.
# Pierwsza tura zadaje pytanie spoza materiałów źródłowych (dbt/Airflow), co powinno skłonić Claude do użycia record_unknown_question zamiast zmyślania odpowiedzi.
# Druga tura, w tej samej historii, podaje dane kontaktowe - to powinno wywołać record_user_details.
# Trzecia tura sprawdza pamięć: pyta o treść pierwszego pytania, mając dostęp tylko do history zbudowanej przez poprzednie dwie tury, bez żadnego dodatkowego mechanizmu pamięci.

with trace("Rozmowa z cyfrowym bliźniakiem"):  # jeden znacznik czasu obejmujący całą trzyturową rozmowę
    answer1, history = run(twin_instructions, "Cześć! Masz doświadczenie z dbt i Airflow w produkcyjnych pipeline'ach?", tools=twin_tools)  # tura 1: pytanie poza materiałami źródłowymi
    print(f"Bliźniak: {answer1}\n")  # odpowiedź po turze 1

    answer2, history = run(twin_instructions, "Chciałabym pogadać o współpracy. Nazywam się Anna Kowalska, mój email to anna.kowalska@firma.pl.", history=history, tools=twin_tools)  # tura 2: przekazujemy history z tury 1 - to jest "pamięć"
    print(f"Bliźniak: {answer2}\n")  # odpowiedź po turze 2

    answer3, history = run(twin_instructions, "Przypomnij, o co pytałam na samym początku?", history=history, tools=twin_tools)  # tura 3: sprawdzamy, czy bliźniak pamięta pytanie z tury 1
    print(f"Bliźniak: {answer3}")  # odpowiedź po turze 3 - powinna odnosić się do pytania o dbt/Airflow

[trace] start: Rozmowa z cyfrowym bliźniakiem
Bliźniak: Cześć! 

Dbt faktycznie znam i pracuję z tym narzędziem — projektowanie pipeline'ów opartych na dbt to część mojego portfolio, szczególnie w kontekście GCP/BigQuery. Tam dbt świetnie się sprawdza, bo naturalnie integruje się z BigQuery.

Jeśli chodzi o **Airflow**, to szczerze mówiąc — nie mam bezpośredniego produkcyjnego doświadczenia z tym narzędziem. To jest ślepa plama w moim CV. W ostatnich latach bardziej skupiałem się na **n8n** do automatyzacji i orchestracji workflow'ów, co ma trochę inny charakter niż Airflow.

Rozumiem, że Airflow to solidny standard do zarządzania DAG'ami w dużych systemach, ale jeśli to dla Ciebie ważne doświadczenie — mogę szczerze powiedzieć, że tutaj bym musiał wdrożyć się właściwie.

**Czemu pytasz?** Jeśli szukasz kogoś do projektu czy współpracy związanej z tymi narzędziami, mogę być otwarty na temat — mogę się tego nauczyć, ale warto żebyśmy byli czystych linii co do tego, na czym stoi moja bie